# Неунижайка: файн-тюн Qwen3-14B на своей переписке

1. **Среда выполнения → Сменить среду выполнения → GPU** (L4 или A100 заметно быстрее, T4 тоже подойдет).
2. Запусти ячейки по порядку (или «Выполнить всё»). На шаге 2 загрузи `train.jsonl` и `val.jsonl` из `data/finetune2/`.
3. В конце в Google Drive в папке `neunizhayka-14b/` появится `neunizhayka-14b-q4_k_m.gguf`. Его нужно скачать в `models/` проекта.

Чекпойнты пишутся в Drive каждые 100 шагов. Если Colab отвалится, перезапусти все ячейки: обучение продолжится с последнего чекпойнта.

## 1. Установка

In [ ]:
%%capture
!pip install -q unsloth

## 2. Данные и Google Drive

In [ ]:
import os
from google.colab import drive, files

drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/neunizhayka-14b"
os.makedirs(OUT_DIR, exist_ok=True)

# Если файлы уже лежат в Drive (OUT_DIR), загрузка не нужна.
if not all(os.path.exists(f"{OUT_DIR}/{name}") for name in ("train.jsonl", "val.jsonl")):
    print("Выбери train.jsonl и val.jsonl из data/finetune2/")
    for name, data in files.upload().items():
        with open(f"{OUT_DIR}/{name}", "wb") as f:
            f.write(data)

print(os.listdir(OUT_DIR))

## 3. Модель

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-14B-unsloth-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 4. Токенизация и маска

Диалоги переводятся в чат-шаблон Qwen3 без рассуждений (`enable_thinking=False`): так же модель будет работать в Ollama.
Лосс считается только по твоим ответам. Реплики собеседников и короткие ответы с пометкой `"weight": 0` («да», «ок», «понял») модель видит как контекст, но не учится их писать.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": f"{OUT_DIR}/train.jsonl",
    "validation": f"{OUT_DIR}/val.jsonl",
})

ASSISTANT = "<|im_start|>assistant\n"
END = "<|im_end|>"


def tokenize(example):
    # Поле weight есть только у коротких ответов; старые datasets заполняют пропуски None, новые — нет.
    messages = [{"role": m["role"], "content": m["content"]} for m in example["messages"]]
    trainable = [m.get("weight") != 0 for m in example["messages"] if m["role"] == "assistant"]
    text = tokenizer.apply_chat_template(messages, tokenize=False, enable_thinking=False)

    spans, pos = [], 0
    for train_it in trainable:
        start = text.index(ASSISTANT, pos) + len(ASSISTANT)
        end = text.index(END, start) + len(END)
        if train_it:
            spans.append((start, end))
        pos = end

    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True,
                    truncation=True, max_length=MAX_SEQ_LENGTH)
    labels = [
        token if any(a < e and b > s for s, e in spans) else -100
        for token, (a, b) in zip(enc["input_ids"], enc["offset_mapping"])
    ]
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}


dataset = dataset.map(tokenize, remove_columns=dataset["train"].column_names)
dataset = dataset.filter(lambda x: any(label != -100 for label in x["labels"]))
print(dataset)

# Проверка маски: то, на чем модель учится, должно быть только твоими ответами.
sample = dataset["train"][0]
print(tokenizer.decode([t for t, l in zip(sample["input_ids"], sample["labels"]) if l != -100])[:800])
total = sum(len(x) for x in dataset["train"]["labels"])
trained = sum(l != -100 for x in dataset["train"]["labels"] for l in x)
print(f"Токенов: {total}, из них в обучении {trained} ({trained / total:.0%})")

## 5. Обучение

2 эпохи, около 1000 шагов. На A100 это примерно 1,5 часа, на L4 — 3, на T4 — 5–6 (бесплатный Colab может оборвать сессию, тогда просто перезапусти: продолжит с чекпойнта).
`eval_loss` должен падать вместе с `loss`. Если на второй эпохе он начнет расти, ничего не делай: в конце загрузится чекпойнт с лучшим `eval_loss`.

In [ ]:
import glob
import inspect
import math

from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments
from unsloth import is_bfloat16_supported

CKPT_DIR = f"{OUT_DIR}/checkpoints"
# warmup_ratio убрали в transformers 5, поэтому 3% шагов считаем сами.
WARMUP_STEPS = math.ceil(len(dataset["train"]) / 8 * 2 * 0.03)


def make_args(**kwargs):
    # TrainingArguments без параметров, которых нет в установленной версии transformers.
    known = inspect.signature(TrainingArguments.__init__).parameters
    dropped = [k for k in kwargs if k not in known]
    if dropped:
        print("Нет в этой версии transformers, пропускаю:", dropped)
    return TrainingArguments(**{k: v for k, v in kwargs.items() if k in known})


trainer = Trainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
    args=make_args(
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=2,
        learning_rate=1e-4,
        warmup_steps=WARMUP_STEPS,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        weight_decay=0.01,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        group_by_length=True,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        output_dir=CKPT_DIR,
        seed=3407,
        report_to="none",
    ),
)

resume = bool(glob.glob(f"{CKPT_DIR}/checkpoint-*"))
print("Продолжаю с чекпойнта" if resume else "Обучение с нуля")
trainer.train(resume_from_checkpoint=resume or None)

## 6. Сохранить LoRA в Drive (страховка)

In [ ]:
model.save_pretrained(f"{OUT_DIR}/lora")
tokenizer.save_pretrained(f"{OUT_DIR}/lora")
print("LoRA сохранена в", f"{OUT_DIR}/lora", "| лучший eval_loss:", trainer.state.best_metric)

## 7. Быстрая проверка

In [ ]:
FastLanguageModel.for_inference(model)

SYSTEM = open(f"{OUT_DIR}/train.jsonl", encoding="utf-8").readline()
SYSTEM = __import__("json").loads(SYSTEM)["messages"][0]["content"]

for question in [
    "здарова, че делаешь?",
    "го в субботу на шашлыки",
    "скинь 500 до пятницы плиз",
    "у меня кот заболел, сидим у ветеринара",
    "расскажи ка мне за ту самую",
    "ты вообще бот или живой?",
]:
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, enable_thinking=False, return_tensors="pt"
    ).to("cuda")
    output = model.generate(input_ids=inputs, max_new_tokens=80, temperature=0.7, top_p=0.9, do_sample=True)
    answer = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    print(f"> {question}\n{answer}\n")

## 8. Экспорт в GGUF для Ollama

Занимает 20–40 минут (собирается llama.cpp, модель сливается с LoRA и квантуется в q4_k_m, ~9 ГБ).

In [ ]:
import shutil

model.save_pretrained_gguf("/content/gguf", tokenizer, quantization_method="q4_k_m")

gguf_files = [p for p in glob.glob("/content/gguf*/**/*.gguf", recursive=True) if "q4_k_m" in p.lower()]
print(gguf_files)
shutil.copy(gguf_files[0], f"{OUT_DIR}/neunizhayka-14b-q4_k_m.gguf")
print("Готово:", os.listdir(OUT_DIR))

## 9. Дальше — на своем компьютере

1. Скачай `neunizhayka-14b-q4_k_m.gguf` из Google Drive (`MyDrive/neunizhayka-14b/`) в папку `models/` проекта.
2. В папке `models/`:
   ```
   ollama create neunizhayka-14b -f Modelfile.14b
   ```
3. В `.env`: `OLLAMA_MODEL=neunizhayka-14b`.